# Crosstab, Method Chaining & Advanced Data Cleaning (5+ Years Interview Guide)
Exhaustive senior guide to contingency tables (pd.crosstab), method chaining with .pipe(), conditional masking (.mask), outlier clipping (.clip), and time-series interpolation.

### Key 5-Year Interview Concepts Covered:
- **Contingency Tables (`pd.crosstab`)**: Frequency distributions, normalization ('index', 'columns', 'all'), and subtotal margins.
- **Method Chaining Architecture (`.pipe()`)**: Writing modular, functional, and testable data transformation pipelines.
- **Conditional Masking (`df.mask()`)**: Inverting `.where()` conditions for clean outlier and anomaly replacement.
- **Outlier Capping (`df.clip()`)**: Winsorizing and bounding numerical extremes without dropping rows.
- **Time-Series Interpolation (`df.interpolate()`)**: Linear, time-weighted, and polynomial imputation of missing signals.

This interactive notebook is fully customized using the Fintech dataset `data/raw_transactions.csv`.

In [ ]:
# Setup imports & load dataset
import pandas as pd
import numpy as np
import os

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path, na_values=['Nan', ''])
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(2)

## Section 1: Contingency Tables with `pd.crosstab`

### Frequency Contingency Tables (`pd.crosstab`)
**Explanation**: `pd.crosstab(index, columns)` computes a frequency distribution table of two or more factors. Setting `margins=True` adds 'All' row and column subtotal sums. Setting `normalize='index'` converts counts into row percentages (e.g. percentage of fraud per card type), which is standard in risk modeling.

**Syntax**: `pd.crosstab(df['card_type'], df['is_fraud'], normalize='index', margins=True) * 100`

In [ ]:
fraud_crosstab = pd.crosstab(df['card_type'], df['is_fraud'], margins=True, normalize='index') * 100
print('Fraud Rate (%) by Card Type:\n', fraud_crosstab.round(2))

### Aggregated Values in Crosstab
**Explanation**: `pd.crosstab()` also supports aggregate functions across third numeric columns: `pd.crosstab(df['region'], df['card_type'], values=df['transaction_amount'], aggfunc='mean')` calculates average amounts per matrix cell.

**Syntax**: `pd.crosstab(df['region'], df['card_type'], values=df['amount'], aggfunc='mean')`

In [ ]:
value_matrix = pd.crosstab(df['region'], df['card_type'], values=df['transaction_amount'], aggfunc='mean').round(2)
print('Average Transaction Amount Matrix:\n', value_matrix)

## Section 2: Method Chaining with `.pipe()`

### Functional Pipeline Architecture (`.pipe()`)
**Explanation**: The `.pipe(func, *args, **kwargs)` method applies a function that takes a DataFrame as its first argument and returns a transformed DataFrame. This allows writing readable, fluent method-chaining pipelines without intermediate temporary variable clutter.

**Syntax**: `df.pipe(clean_data).pipe(calculate_metrics).pipe(filter_active)`

In [ ]:
def clean_regions(df_in):
    df_out = df_in.copy()
    df_out['region'] = df_out['region'].str.strip().str.capitalize()
    return df_out

def filter_valid_amounts(df_in, min_val=10.0):
    return df_in[df_in['transaction_amount'] >= min_val]

pipeline_df = (
    df
    .pipe(clean_regions)
    .pipe(filter_valid_amounts, min_val=50.0)
)
print('Pipeline Result Shape:', pipeline_df.shape)

## Section 3: Advanced Data Cleaning: Mask, Clip & Interpolate

### Conditional Masking (`df.mask()`)
**Explanation**: `df.mask(condition, other=replacement)` is the exact logical opposite of `.where()`. It replaces values where the condition is `True` with `other` (defaults to `NaN`). It is ideal for masking sensitive fields or nullifying anomaly outliers.

**Syntax**: `df['amount'].mask(df['amount'] > 1400, np.nan)`

In [ ]:
sample_amounts = df['transaction_amount'].head(5).copy()
masked = sample_amounts.mask(sample_amounts > 500, -999.0)
print('Original:\n', sample_amounts)
print('\nMasked (>500 replaced with -999):\n', masked)

### Outlier Capping & Winsorization (`df.clip()`)
**Explanation**: `df.clip(lower=min_val, upper=max_val)` clamps numeric values to specified lower and upper thresholds in-place in C. Unlike dropping rows, clipping preserves sample size while mitigating extreme outlier distortion in linear models.

**Syntax**: `df['amount'].clip(lower=50.0, upper=1200.0)`

In [ ]:
clipped_amounts = df['transaction_amount'].clip(lower=100.0, upper=1000.0)
print('Original Min/Max:', df['transaction_amount'].min(), df['transaction_amount'].max())
print('Clipped Min/Max:', clipped_amounts.min(), clipped_amounts.max())

### Time-Series Missing Value Interpolation (`df.interpolate()`)
**Explanation**: `df.interpolate(method='linear')` estimates missing `NaN` values using mathematical interpolation (linear, time-weighted, polynomial) based on neighboring points. In financial time series, interpolation reconstructs missing market quotes without introducing step discontinuities.

**Syntax**: `df['amount'].interpolate(method='linear')`

In [ ]:
series_with_nans = pd.Series([100.0, np.nan, np.nan, 400.0, 500.0])
interpolated = series_with_nans.interpolate(method='linear')
print('Original:\n', series_with_nans)
print('\nLinearly Interpolated:\n', interpolated)

## Section: Senior Fintech Interview Questions (5+ Years Experience)

### Q1: Device Type vs Fraud Rate Crosstab with Subtotals
**Explanation**: Generate a crosstab measuring fraud incidence (`is_fraud`) across distinct `device_type` values, showing normalized percentages and total row counts.

**Syntax**: `pd.crosstab(df['device_type'], df['is_fraud'], normalize='index', margins=True) * 100`

In [ ]:
device_fraud = pd.crosstab(df['device_type'], df['is_fraud'], normalize='index', margins=True) * 100
print('Device Type Fraud Breakdown (%):\n', device_fraud.round(2))